<a href="https://colab.research.google.com/github/kinemax-core/Genomic-data-science-R/blob/main/dnaseq_analyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import collections
import os

def parse_fasta(filepath):
    """
    (1) & (2) Parses a multi-FASTA file in Google Colab.
    Returns a dictionary mapping identifiers to their complete concatenated sequences.
    """
    records = {}
    current_id = None
    current_seq = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_id:
                    records[current_id] = "".join(current_seq)
                # Extract the pure identifier word immediately following '>'
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        # Handle the very last sequence record in the file
        if current_id:
            records[current_id] = "".join(current_seq)

    return records

def analyze_sequences(records):
    """
    (1) & (2) Computes record count, lengths, and identifies longest/shortest entries.
    """
    print(f"==================================================")
    print(f"📢 EXAM PART 1 & 2: RECORD AND SEQUENCE STATISTICS")
    print(f"==================================================")
    total_records = len(records)
    print(f"👉 Q1: Total number of records in file: {total_records}\n")

    lengths = {header: len(seq) for header, seq in records.items()}
    max_len = max(lengths.values())
    min_len = min(lengths.values())

    longest_ids = [k for k, v in lengths.items() if v == max_len]
    shortest_ids = [k for k, v in lengths.items() if v == min_len]

    print(f"👉 Q2: Sequence Length Parameters:")
    print(f"  • Shortest sequence length: {min_len} bp")
    print(f"    Identifiers matching shortest: {shortest_ids}\n")
    print(f"  • Longest sequence length: {max_len} bp")
    print(f"    Identifiers matching longest: {longest_ids}\n")


def find_orfs_in_frame(sequence, frame):
    """
    (3) Scans a forward reading frame (1, 2, or 3) for valid ORFs.
    Ensures an ORF starts with ATG and explicitly terminates at an in-frame stop codon.
    """
    start_codons = {"ATG"}
    stop_codons = {"TAA", "TAG", "TGA"}
    orfs = []

    # 1-based frames map to 0-based Python sequence indexing
    start_index = frame - 1
    seq_len = len(sequence)

    i = start_index
    while i < seq_len - 2:
        codon = sequence[i:i+3].upper()

        if codon in start_codons:
            # Look ahead in triplets for a valid, matching stop codon
            for j in range(i + 3, seq_len - 2, 3):
                next_codon = sequence[j:j+3].upper()
                if next_codon in stop_codons:
                    orf_seq = sequence[i:j+3]
                    start_pos = i + 1  # 1-indexed character position for your quiz entry
                    orf_len = len(orf_seq)
                    orfs.append((start_pos, orf_len, orf_seq))
                    break # Found terminal point; stop searching for this specific ATG
        i += 3 # Move to the next triplet window position

    return orfs

def analyze_orfs_for_all_frames(records):
    """
    (3) Iterates through frames 1, 2, and 3 separately so you can fill out any quiz prompt.
    """
    print(f"==================================================")
    print(f"📢 EXAM PART 3: OPEN READING FRAME (ORF) ANALYSIS")
    print(f"==================================================")

    for frame in [1, 2, 3]:
        global_max_len = 0
        global_max_id = None
        global_max_start = None

        print(f"👉 Checking Forward Reading Frame {frame}:")
        for header, seq in records.items():
            found_orfs = find_orfs_in_frame(seq, frame)
            if found_orfs:
                # Find local longest inside this specific sequence header
                local_longest = max(found_orfs, key=lambda x: x[1])
                if local_longest[1] > global_max_len:
                    global_max_len = local_longest[1]
                    global_max_id = header
                    global_max_start = local_longest[0]

        if global_max_id:
            print(f"  • Longest ORF length: {global_max_len} bp")
            print(f"  • Found in Sequence ID: {global_max_id}")
            print(f"  • 1-indexed Start Position: {global_max_start}\n")
        else:
            print("  • No valid ORFs found in this frame.\n")


def find_most_frequent_repeats(records, length_list):
    """
    (4) Counts structural repeats of varying length 'n' allowing window shifts/overlaps.
    """
    print(f"==================================================")
    print(f"📢 EXAM PART 4: STRUCTURAL REPEAT ANALYSIS")
    print(f"==================================================")

    for n in length_list:
        repeat_counts = collections.defaultdict(int)

        for seq in records.values():
            for i in range(len(seq) - n + 1):
                kmer = seq[i:i+n]
                repeat_counts[kmer] += 1

        # Only preserve tracking substrings that show up more than once
        valid_repeats = {k: v for k, v in repeat_counts.items() if v > 1}

        print(f"👉 Checking Repeats of Length n = {n}:")
        if not valid_repeats:
            print("  • No recurring repeat sequences found for this length.\n")
            continue

        max_frequency = max(valid_repeats.values())
        most_frequent_kmers = [k for k, v in valid_repeats.items() if v == max_frequency]

        print(f"  • Total unique repeat sequences: {len(valid_repeats)}")
        print(f"  • Maximum occurrence count: {max_frequency}")
        print(f"  • Most frequent structural pattern(s): {most_frequent_kmers}\n")


# Standard Colab runtime configuration execution block
if __name__ == "__main__":
    colab_filename = "dna2.fasta"

    if os.path.exists(colab_filename):
        # 1. Parse and compile genomics dictionaries
        dna_records = parse_fasta(colab_filename)

        # 2. Extract structural characteristics (Q1 & Q2 answers)
        analyze_sequences(dna_records)

        # 3. Extract Open Reading Frames across all requested frames (Q3 answers)
        analyze_orfs_for_all_frames(dna_records)

        # 4. Extract structural overlapping repeats (Q4 answers)
        # Note: You can add or modify lengths inside this list (e.g., [6, 7, 12])
        # depending on what specific integer length your active quiz question asks for!
        find_most_frequent_repeats(dna_records, length_list=[7, 12])

    else:
        print(f"🛑 File Error: '{colab_filename}' was not found in your main environment directory.")
        print("Please click the folder icon on the left panel of Google Colab and upload your file.")

In [ ]:
## global alignment
def global_alignment(seq1, seq2, match_score = 1, mismatch_score = -1, gap_penalty = -1 ):
  m, n = len(seq1), len(seq2)

  score_matrix = [[0]*(n+1) for _ in range(m+1)]
  for i in range(m+1):
    score_matrix[i][0] = i * gap_penalty
  for j in range(n+1):
    score_matrix[0][j] = j * gap_penalty


  for i in range(1, m+1):
    for j in range(1, n+1):
      ## score for diagonal step
      if seq1[i-1] == seq2[j-1]:
        diagonal = score_matrix[i-1][j-1] + match_score
      else:
        diagona = score_matrix[i-1][j-i] + mismatch_score


      up = score_matrix[i-1][j] + gap_penalty
      left = score_matrix[i][j-1] + gap_penalty

      score_matrix[i][j] = max(diagonal, up, left)


  aligned_seq1 = []
  aligned_seq2 = []

  i, j = m, n
  while i > 0 and j > 0:
    current_score = score_matrix[i][j]
    if i > 0 and j > 0 and((seq1[i-1] == seq2[j-1] and current_score == score_matrix[i-1][j-1] + match_score) or(seq1[i-1] != seq2[j-1] and current_score == score_matrix[i-1][j-1] + mismatch_score)):
      aligned_seq1.append(seq1[i-1])
      aligned_seq2.append(seq2[j-1])
      i -= 1
      j -= 1
    elif i > 0 and current_score == score_matrix[i-1][j] + gap_penalty:
      aligned_seq1.append(seq1[i-1])
      aligned_seq2.append('-')
      i -= 1

    else:
      aligned_seq1.append('-')
      aligned_seq2.append(seq2[j-1])
      j -= 1


  final_seq = "".join(reversed(aligned_seq1))
  final_seq2 = "".join(reversed(aligned_seq2))

  return score_matrix[m][n], final_seq, final_seq2



seqA = "ATGCGTACGGTCACC"
seqB = "ATGCCGTTGCCAGTT"
max_score, alignment1, alignment2 = global_alignment(seqA, seqB)
print(f"Alignment Score: {max_score}")
print(f"Aligned Sequence 1: {alignment1}")
print(f"Aligned Sequence 2: {alignment2}")

